In [2]:
"""
Airtel Money ZMK Statement, CSV Extractor
"""

import re, csv, pdfplumber
import pandas as pd

SKIP_LINES = [
    "AIRTEL MONEY STATEMENT", "Customer Name:", "Mobile Number:",
    "Email Address:", "Statement Period:", "Request Date:", "SUMMARY",
    "DETAILED STATEMENT", "Transaction ID", "Transaction Date",
    "Opening Balance", "Closing Balance", "Total Credit", "Total Debit",
    "www.airtel.co.zm", "Disclaimer", "For self-service", "https://",
    "Transaction\nAmount", "Credit/Debit", "Amount"
]

HEADERS = ["TXN_DATE", "VALUE_DATE", "DESCRIPTION", "MONEY_OUT_OR_IN", "BALANCE", "TXN_TYPE"]

PATTERN = re.compile(
    r"^(\S+)\s+"
    r"(\d{2}-\d{2}-\d{2})\s+"
    r"\d{2}:\d{2}\s+[AP]M\s+"
    r"(.+?)\s+"
    r"Transaction Successful\s+"
    r"([\d,]+\.?\d*)\s+"
    r"(Credit|Debit)\s+"
    r"([\d,]+\.?\d*)$"
)


def format_date(raw):
    day, month, year = raw.split("-")
    return f"{int(day)}/{int(month)}/20{year}"


def clean_amount(value):
    return float(value.replace(",", "").replace("Zmk", "").strip())


def read_pdf(pdf_path):
    lines = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            lines.extend((page.extract_text() or "").splitlines())
    return lines


def get_metadata(lines):
    def find(label):
        for line in lines:
            if label in line:
                return line.split(label)[-1].strip()
        return ""
    return {
        "account_owner":    find("Customer Name:"),
        "account_no":       find("Mobile Number:"),
        "email":            find("Email Address:"),
        "statement_period": find("Statement Period:"),
        "request_date":     find("Request Date:"),
        "currency":         "ZMK",
        "opening":          clean_amount(find("Opening Balance: Zmk")),
        "closing":          clean_amount(find("Closing Balance: Zmk")),
        "total_credit":     clean_amount(find("Total Credit: Zmk")),
        "total_debit":      clean_amount(find("Total Debit: Zmk")),
    }


def get_transactions(lines):
    transactions = []
    for line in lines:
        line  = line.strip()
        match = PATTERN.match(line)
        if match:
            txn_id, date, desc, amount, cr_dr, balance = match.groups()
            transactions.append({
                "TXN_DATE":        format_date(date),
                "VALUE_DATE":      format_date(date),
                "DESCRIPTION":     desc.strip(),
                "MONEY_OUT_OR_IN": clean_amount(amount),
                "BALANCE":         clean_amount(balance),
                "TXN_TYPE":        "CR" if cr_dr == "Credit" else "DR",
            })
        elif transactions and line and not any(s in line for s in SKIP_LINES):
            if not line[0].isdigit() and not line.startswith("http"):
                transactions[-1]["DESCRIPTION"] += " " + line

    if not transactions:
        raise ValueError("No transactions found — check the PDF format.")

    df = pd.DataFrame(transactions)
    df["MONEY_OUT_OR_IN"] = pd.to_numeric(df["MONEY_OUT_OR_IN"], errors="coerce")
    df["BALANCE"]         = pd.to_numeric(df["BALANCE"],         errors="coerce")
    return df


def write_csv(df, info, output_path):
    metadata = [
        "AccountType:MOBILE MONEY",
        f"StatementPeriod:{info['statement_period']}",
        f"StatementStartDate:{df['TXN_DATE'].iloc[0]}",
        f"StatementEndDate:{df['TXN_DATE'].iloc[-1]}",
        f"AccountNo:{info['account_no']}",
        f"Currency:{info['currency']}",
        f"AccountOwner:{info['account_owner']}",
        f"Email:{info['email']}",
        f"Opening:{info['opening']}",
        f"Closing:{info['closing']}",
        f"TotalCredit:{info['total_credit']}",
        f"TotalDebit:{info['total_debit']}",
        f"RequestDate:{info['request_date']}",
    ]
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([""] + HEADERS)
        for i, item in enumerate(metadata):
            writer.writerow([i, "", "", item, "", "", ""])
        writer.writerow([13] + HEADERS)
        for i, row in df.iterrows():
            writer.writerow([i + 14] + row.tolist())


def run(pdf_path, output_path):
    lines = read_pdf(pdf_path)
    df    = get_transactions(lines)
    info  = get_metadata(lines)
    write_csv(df, info, output_path)
    print(f"Done — {len(df)} transactions saved to {output_path}")


run("Zmk_Statemment.pdf", "Zmk.csv")

Done — 191 transactions saved to Zmk.csv
